In [1]:
# === INFERENCE NOTEBOOK ===
import xgboost as xgb
import pandas as pd
import numpy as np
import pickle

MODEL_PATH     = "models/baseline_xgboost.json"
ARTIFACTS_PATH = "models/preprocessing_info.pkl"
# CSV_PATH = "data/flows.csv"   # <-- The input 80 feature network flow data
CSV_PATH = "hulk_test_flows.csv"   # <-- The input 80 feature network flow data


In [2]:
# 1. Load model and artifacts
model = xgb.XGBClassifier()
model.load_model(MODEL_PATH)

with open(ARTIFACTS_PATH, "rb") as f:
    artifacts = pickle.load(f)

le_target    = artifacts["target_encoder"]      # was "label_encoder"
feature_cols = artifacts["feature_columns"]     # was "feature_cols"

/home/cpre560/miniconda3/envs/cpre560/lib/python3.10/site-packages/sklearn/base.py:442: InconsistentVersionWarning: Trying to unpickle estimator LabelEncoder from version 1.8.0 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [3]:
with open("rename_map.pkl", "rb") as file:
    rename_map = pickle.load(file)
# print(type(rename_map),rename_map)

In [4]:
# 2. Load and preprocess input CSV
df = pd.read_csv(CSV_PATH)
df.columns = df.columns.str.strip()
df.replace([np.inf, -np.inf], np.nan, inplace=True)


df.rename(columns=rename_map, inplace=True)

# CICIDS2017 has a duplicate 'Fwd Header Length' column recorded as 'Fwd Header Length.1'
# The model expects it — just copy the value from the renamed column
df['Fwd Header Length.1'] = df['Fwd Header Length']

# Verify all expected features are now present
missing = [c for c in feature_cols if c not in df.columns]
print(f"Still missing after rename: {missing if missing else 'None — all good!'}")

Still missing after rename: None — all good!


In [5]:
# 2. Load and preprocess input CSV
df = pd.read_csv(CSV_PATH)
df.columns = df.columns.str.strip()
df.replace([np.inf, -np.inf], np.nan, inplace=True)

# Add our rename label mapping to the df
df.rename(columns=rename_map, inplace=True)
df['Fwd Header Length.1'] = df['Fwd Header Length']

df.dropna(subset=feature_cols, inplace=True)
X = df[feature_cols]

In [6]:
# 3. Run inference
y_pred       = model.predict(X)
y_pred_proba = model.predict_proba(X)
labels       = le_target.inverse_transform(y_pred)


In [7]:
# 4. Output results
results = df[feature_cols].copy()
results["Predicted_Label"]    = labels
results["Confidence"]         = y_pred_proba.max(axis=1).round(4)
results["Predicted_Class_ID"] = y_pred

print(results[["Predicted_Label", "Confidence"]].value_counts())
print(results[["Predicted_Label", "Confidence"]].head(20))

results.to_csv("inference_results.csv", index=False)

Predicted_Label  Confidence
BENIGN           0.9999        25416
                 1.0000         1417
                 0.9994          483
                 0.9998          345
                 0.9995          282
                 0.9857           40
                 0.9915           29
                 0.9979           21
                 0.9920           19
                 0.9988           18
                 0.9875           10
                 0.9962            8
                 0.9992            8
                 0.9981            7
                 0.9913            4
                 0.9961            3
                 0.9996            3
                 0.9978            2
                 0.9958            2
                 0.9993            2
                 0.9997            2
                 0.9912            2
                 0.9971            1
                 0.9970            1
                 0.9968            1
                 0.9989            1
          

In [9]:
import pickle
with open("models/preprocessing_info.pkl", "rb") as f:
    artifacts = pickle.load(f)

print(type(artifacts))
print(artifacts.keys() if hasattr(artifacts, 'keys') else dir(artifacts))

<class 'dict'>
dict_keys(['categorical_encoders', 'target_encoder', 'feature_columns', 'categorical_columns'])


/home/cpre560/miniconda3/envs/cpre560/lib/python3.10/site-packages/sklearn/base.py:442: InconsistentVersionWarning: Trying to unpickle estimator LabelEncoder from version 1.8.0 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [10]:
import pandas as pd
df = pd.read_csv("data/flows.csv")
df.columns = df.columns.str.strip()
print(df.columns.tolist())
print(f"\nShape: {df.shape}")

['src_ip', 'dst_ip', 'src_port', 'Destination Port', 'protocol', 'timestamp', 'Flow Duration', 'Flow Bytes/s', 'Flow Packets/s', 'Fwd Packets/s', 'Bwd Packets/s', 'Total Fwd Packets', 'Total Backward Packets', 'Total Length of Fwd Packets', 'Total Length of Bwd Packets', 'Fwd Packet Length Max', 'Fwd Packet Length Min', 'Fwd Packet Length Mean', 'Fwd Packet Length Std', 'Bwd Packet Length Max', 'Bwd Packet Length Min', 'Bwd Packet Length Mean', 'Bwd Packet Length Std', 'Max Packet Length', 'Min Packet Length', 'Packet Length Mean', 'Packet Length Std', 'Packet Length Variance', 'Fwd Header Length', 'Bwd Header Length', 'min_seg_size_forward', 'act_data_pkt_fwd', 'Flow IAT Mean', 'Flow IAT Max', 'Flow IAT Min', 'Flow IAT Std', 'Fwd IAT Total', 'Fwd IAT Max', 'Fwd IAT Min', 'Fwd IAT Mean', 'Fwd IAT Std', 'Bwd IAT Total', 'Bwd IAT Max', 'Bwd IAT Min', 'Bwd IAT Mean', 'Bwd IAT Std', 'Fwd PSH Flags', 'bwd_psh_flags', 'Fwd URG Flags', 'bwd_urg_flags', 'FIN Flag Count', 'SYN Flag Count', 'RST